#Coding Marathon 1: Weather and Trip Planner (prerequisite)

##1: Pair workflow and plan

**Hoang Vu & Sachin Bhandari**

##2: Prepare Colab and Groq access

In [1]:
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
print("Key loaded successfully:", api_key is not None)

Key loaded successfully: True


In [2]:
%pip -q install requests groq langchain-core

In [3]:
import os
from google.colab import userdata
from groq import Groq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
assert os.environ["GROQ_API_KEY"], "Enable GROQ_API_KEY in Colab Secrets."

GROQ_CLIENT = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL_ID = "qwen/qwen3.8-27b"  # Same hosted-Qwen model as the morning activity.

available_ids = {item.id for item in GROQ_CLIENT.models.list().data}
assert MODEL_ID in available_ids, (
    "Selected model unavailable. Check the available tool-capable models."
)
print("Groq client and selected model ready.")

Groq client and selected model ready.


##3: Explore the supplied attractions database

###3.1: Create marathon_db.py

In [4]:
%%writefile marathon_db.py
"""Read-only attractions catalog for the weather and trip-planner marathon."""

import sqlite3
from pathlib import Path

DB_PATH = Path("attractions_sweden_germany.sqlite")

# sample records; coordinates are approximate.
ATTRACTIONS = [
    (1, "Djurgården", "Stockholm", "Sweden", "park", 1, 59.325, 18.103,
     "Large area for outdoor walks and sightseeing."),
      (2, "Skansen Open-Air Museum", "Stockholm", "Sweden", "museum", 1,
     59.324, 18.101, "Open-air cultural attractions and historical buildings."),
    (3, "Vasa Museum", "Stockholm", "Sweden", "museum", 0,
     59.328, 18.091, "Indoor museum with a historic ship."),
    (4, "Slottsskogen", "Gothenburg", "Sweden", "park", 1,
     57.683, 11.947, "Large city park with walking paths."),
    (5, "Gothenburg Botanical Garden", "Gothenburg", "Sweden", "garden", 1,
     57.681, 11.953, "Outdoor botanical garden and walking areas."),
    (6, "Universeum", "Gothenburg", "Sweden", "museum", 0,
     57.695, 11.989, "Indoor science and nature exhibits."),
    (7, "Tiergarten", "Berlin", "Germany", "park", 1,
     52.515, 13.351, "Large green space in central Berlin."),
    (8, "Tempelhofer Feld", "Berlin", "Germany", "park", 1,
     52.473, 13.405, "Open recreational space on a former airfield."),
    (9, "German Museum of Technology", "Berlin", "Germany", "museum", 0,
     52.498, 13.377, "Indoor technology collections."),
    (10, "English Garden", "Munich", "Germany", "park", 1,
     48.152, 11.593, "Outdoor green space and walking routes."),
    (11, "Olympiapark", "Munich", "Germany", "park", 1,
     48.174, 11.549, "Outdoor recreation area and landscaped grounds."),
    (12, "Deutsches Museum", "Munich", "Germany", "museum", 0,
     48.130, 11.583, "Indoor science and technology museum."),
]


def initialize_db():
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS attractions (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                city TEXT NOT NULL,
                country TEXT NOT NULL,
                category TEXT NOT NULL,
                is_outdoor INTEGER NOT NULL CHECK(is_outdoor IN (0, 1)),
                latitude REAL NOT NULL,
                longitude REAL NOT NULL,
                description TEXT NOT NULL
            )
        """)

        connection.executemany("""
            INSERT OR IGNORE INTO attractions
            (id, name, city, country, category, is_outdoor,
             latitude, longitude, description)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, ATTRACTIONS)

    return len(ATTRACTIONS)


def find_attractions(city, category=None, outdoor_only=True, limit=5):
    """Search the local catalog with parameterized SQL."""

    if not city or not city.strip():
        raise ValueError("city is required")

    if not 1 <= limit <= 12:
        raise ValueError("limit must be from 1 to 12")

    sql = "SELECT * FROM attractions WHERE lower(city) = lower(?)"
    arguments = [city.strip()]

    if outdoor_only:
        # TODO 1A: Restrict results to is_outdoor = 1.
        sql += " AND is_outdoor = ?"
        arguments.append(1)

    if category:
        # TODO 1B: Add a parameterized category filter.
        sql += " AND lower(category) = lower(?)"
        arguments.append(category.strip())

    sql += " ORDER BY name LIMIT ?"
    arguments.append(limit)

    with sqlite3.connect(DB_PATH) as connection:
        connection.row_factory = sqlite3.Row
        rows = connection.execute(sql, arguments).fetchall()

    return [dict(row) for row in rows]


if __name__ == "__main__":
    initialize_db()
    print("Database:", DB_PATH)
    print("Berlin:", find_attractions("Berlin"))

Overwriting marathon_db.py


In [5]:
!python marathon_db.py

Database: attractions_sweden_germany.sqlite
Berlin: [{'id': 8, 'name': 'Tempelhofer Feld', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.473, 'longitude': 13.405, 'description': 'Open recreational space on a former airfield.'}, {'id': 7, 'name': 'Tiergarten', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.515, 'longitude': 13.351, 'description': 'Large green space in central Berlin.'}]


###3.2: TODO 1: Complete the SQL filters

In [6]:
%%writefile marathon_db.py
"""Read-only attractions catalog for the weather and trip-planner marathon."""

import sqlite3
from pathlib import Path

DB_PATH = Path("attractions_sweden_germany.sqlite")

# sample records; coordinates are approximate.
ATTRACTIONS = [
    (1, "Djurgården", "Stockholm", "Sweden", "park", 1, 59.325, 18.103,
     "Large area for outdoor walks and sightseeing."),
      (2, "Skansen Open-Air Museum", "Stockholm", "Sweden", "museum", 1,
     59.324, 18.101, "Open-air cultural attractions and historical buildings."),
    (3, "Vasa Museum", "Stockholm", "Sweden", "museum", 0,
     59.328, 18.091, "Indoor museum with a historic ship."),
    (4, "Slottsskogen", "Gothenburg", "Sweden", "park", 1,
     57.683, 11.947, "Large city park with walking paths."),
    (5, "Gothenburg Botanical Garden", "Gothenburg", "Sweden", "garden", 1,
     57.681, 11.953, "Outdoor botanical garden and walking areas."),
    (6, "Universeum", "Gothenburg", "Sweden", "museum", 0,
     57.695, 11.989, "Indoor science and nature exhibits."),
    (7, "Tiergarten", "Berlin", "Germany", "park", 1,
     52.515, 13.351, "Large green space in central Berlin."),
    (8, "Tempelhofer Feld", "Berlin", "Germany", "park", 1,
     52.473, 13.405, "Open recreational space on a former airfield."),
    (9, "German Museum of Technology", "Berlin", "Germany", "museum", 0,
     52.498, 13.377, "Indoor technology collections."),
    (10, "English Garden", "Munich", "Germany", "park", 1,
     48.152, 11.593, "Outdoor green space and walking routes."),
    (11, "Olympiapark", "Munich", "Germany", "park", 1,
     48.174, 11.549, "Outdoor recreation area and landscaped grounds."),
    (12, "Deutsches Museum", "Munich", "Germany", "museum", 0,
     48.130, 11.583, "Indoor science and technology museum."),
]


def initialize_db():
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS attractions (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                city TEXT NOT NULL,
                country TEXT NOT NULL,
                category TEXT NOT NULL,
                is_outdoor INTEGER NOT NULL CHECK(is_outdoor IN (0, 1)),
                latitude REAL NOT NULL,
                longitude REAL NOT NULL,
                description TEXT NOT NULL
            )
        """)

        connection.executemany("""
            INSERT OR IGNORE INTO attractions
            (id, name, city, country, category, is_outdoor,
             latitude, longitude, description)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, ATTRACTIONS)

    return len(ATTRACTIONS)


def find_attractions(city, category=None, outdoor_only=True, limit=5):
    """Search the local catalog with parameterized SQL."""

    if not city or not city.strip():
        raise ValueError("city is required")

    if not 1 <= limit <= 12:
        raise ValueError("limit must be from 1 to 12")

    sql = "SELECT * FROM attractions WHERE lower(city) = lower(?)"
    arguments = [city.strip()]

    if outdoor_only:
        sql += " AND is_outdoor = ?"
        arguments.append(1)

    if category:
        sql += " AND lower(category) = lower(?)"
        arguments.append(category.strip())

    sql += " ORDER BY name LIMIT ?"
    arguments.append(limit)

    with sqlite3.connect(DB_PATH) as connection:
        connection.row_factory = sqlite3.Row
        rows = connection.execute(sql, arguments).fetchall()

    return [dict(row) for row in rows]

if __name__ == "__main__":
    initialize_db()
    print("Database:", DB_PATH)
    print("Berlin:", find_attractions("Berlin"))

Overwriting marathon_db.py


In [7]:
!python marathon_db.py

Database: attractions_sweden_germany.sqlite
Berlin: [{'id': 8, 'name': 'Tempelhofer Feld', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.473, 'longitude': 13.405, 'description': 'Open recreational space on a former airfield.'}, {'id': 7, 'name': 'Tiergarten', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.515, 'longitude': 13.351, 'description': 'Large green space in central Berlin.'}]


In [8]:
from marathon_db import initialize_db, find_attractions

initialize_db()

berlin_parks = find_attractions(
    "Berlin",
    category="park",
    outdoor_only=True
)

assert berlin_parks, "Expected outdoor parks in Berlin."
assert all(row["is_outdoor"] == 1 for row in berlin_parks)
assert all(row["category"] == "park" for row in berlin_parks)

stockholm_outdoor = find_attractions(
    "Stockholm",
    outdoor_only=True
)

assert stockholm_outdoor
assert all(row["is_outdoor"] == 1 for row in stockholm_outdoor)

assert find_attractions("Madrid") == []

print("Database filters passed.")

Database filters passed.


# Step-4

Create marathon_api.py

In [9]:

%%writefile marathon_api.py
"""Open-Meteo daily forecast helper for future trip planning."""
from datetime import datetime, timezone
import requests

FORECAST_URL = "https://api.open-meteo.com/v1/forecast"


def get_forecast(latitude, longitude, days=7):
    """Return compact daily forecasts for a location."""

    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        raise ValueError("Invalid latitude or longitude")

    if not 1 <= days <= 7:
        raise ValueError("days must be from 1 to 7")

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "precipitation_probability_max,wind_speed_10m_max",
        "forecast_days": days,
        "timezone": "auto",
        "wind_speed_unit": "kmh"
    }

    response = requests.get(
        FORECAST_URL,
        params=params,
        timeout=20
    )
    response.raise_for_status()

    data = response.json()
    daily = data["daily"]

    observations = []

    for index, date_string in enumerate(daily["time"]):
        observations.append({
            "date": date_string,
            "rain_probability_max_pct":
                daily["precipitation_probability_max"][index],
            "wind_speed_max_kmh":
                daily["wind_speed_10m_max"][index]
        })

    return {
        "timezone": data.get("timezone"),
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "source": "Open-Meteo Forecast API",
        "days": observations
    }


if __name__ == "__main__":
    sample = get_forecast(52.515, 13.351, days=3)
    print(sample)


Overwriting marathon_api.py


In [10]:

!python marathon_api.py


{'timezone': 'Europe/Berlin', 'retrieved_at_utc': '2026-11-12T12:42:38.056016+00:00', 'source': 'Open-Meteo Forecast API', 'days': [{'date': '2026-11-12', 'rain_probability_max_pct': 10, 'wind_speed_max_kmh': 12.1}, {'date': '2026-11-13', 'rain_probability_max_pct': 8, 'wind_speed_max_kmh': 22.4}, {'date': '2026-11-14', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 14.9}]}


Test the forecast function

In [11]:

from marathon_api import get_forecast

forecast = get_forecast(52.515, 13.351, days=3)

assert len(forecast["days"]) == 3
assert forecast["timezone"]

for day in forecast["days"]:
    assert isinstance(
        day["rain_probability_max_pct"], (int, float)
    )
    assert isinstance(
        day["wind_speed_max_kmh"], (int, float)
    )
    assert 0 <= day["rain_probability_max_pct"] <= 100
    assert day["wind_speed_max_kmh"] >= 0

print("Forecast extraction passed.")
print(forecast)


Forecast extraction passed.
{'timezone': 'Europe/Berlin', 'retrieved_at_utc': '2026-11-12T12:42:38.544771+00:00', 'source': 'Open-Meteo Forecast API', 'days': [{'date': '2026-11-12', 'rain_probability_max_pct': 10, 'wind_speed_max_kmh': 12.1}, {'date': '2026-11-13', 'rain_probability_max_pct': 8, 'wind_speed_max_kmh': 22.4}, {'date': '2026-11-14', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 14.9}]}


# Step-5


Create marathon_tools.py

In [12]:

%%writefile marathon_tools.py
"""LangChain wrappers around the approved database and API functions."""

from langchain_core.tools import tool
from marathon_db import find_attractions
from marathon_api import get_forecast


@tool
def find_attractions_tool(
    city: str,
    category: str | None = None,
    outdoor_only: bool = True,
    limit: int = 5,
) -> list[dict]:
    """Search seeded attractions in Stockholm, Gothenburg, Berlin, or Munich.

    Use for choosing real candidate places. Returns names, approximate
    coordinates, categories, indoor/outdoor flags, and descriptions.
    Does not provide opening hours or real-time availability.
    """
    return find_attractions(city, category, outdoor_only, limit)


@tool
def get_forecast_tool(
    latitude: float,
    longitude: float,
    days: int = 7
) -> dict:
    """Get a future daily forecast for coordinates from Open-Meteo.

    Returns local dates, daily maximum precipitation probability (%),
    and daily maximum 10-m wind speed (km/h). Call after obtaining
    coordinates from a returned attractions database row.
    """
    return get_forecast(latitude, longitude, days)


APPROVED_TOOLS = [
    find_attractions_tool,
    get_forecast_tool
]

TOOL_REGISTRY = {
    item.name: item for item in APPROVED_TOOLS
}


def groq_schema(item):
    """Convert a LangChain tool into a Groq-compatible function schema."""
    return {
        "type": "function",
        "function": {
            "name": item.name,
            "description": item.description,
            "parameters": item.args_schema.model_json_schema()
        }
    }


TOOL_SCHEMAS = [
    groq_schema(item) for item in APPROVED_TOOLS
]


Overwriting marathon_tools.py


Test both tools without an LLM

In [13]:

from marathon_tools import (
    find_attractions_tool,
    get_forecast_tool,
    TOOL_SCHEMAS
)

# Test the database tool
attractions = find_attractions_tool.invoke({
    "city": "Stockholm",
    "outdoor_only": True,
    "limit": 3
})

assert attractions
assert all(place["is_outdoor"] == 1 for place in attractions)

# Use a real returned attraction to test the forecast tool
place = attractions[0]

weather = get_forecast_tool.invoke({
    "latitude": place["latitude"],
    "longitude": place["longitude"],
    "days": 3
})

assert len(weather["days"]) == 3
assert all(
    day["rain_probability_max_pct"] is not None
    for day in weather["days"]
)

print(
    "Registered tool names:",
    [item["function"]["name"] for item in TOOL_SCHEMAS]
)
print("Attraction:", place["name"])
print("Forecast sample:", weather["days"][0])
print("Both tool tests passed.")


Registered tool names: ['find_attractions_tool', 'get_forecast_tool']
Attraction: Djurgården
Forecast sample: {'date': '2026-11-12', 'rain_probability_max_pct': 100, 'wind_speed_max_kmh': 23.0}
Both tool tests passed.


# Step-6


Install and check the dependencies

In [14]:
%pip -q install groq langchain-core requests

In [15]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get("GROQ_API_KEY")
assert api_key, "Add GROQ_API_KEY in Colab Secrets first."

client = Groq(api_key=api_key)

# Check whether the assignment's suggested model is available.
models = client.models.list()
available_models = [model.id for model in models.data]

MODEL = "qwen/qwen3.8-27b"

if MODEL in available_models:
    print("Model available:", MODEL)
else:
    print("Suggested model is not available. Available Qwen models:")
    print([name for name in available_models if "qwen" in name.lower()])

Model available: qwen/qwen3.8-27b


In [16]:

%%writefile marathon_agent.py
import json
from google.colab import userdata
from groq import Groq

from marathon_tools import TOOL_SCHEMAS, TOOL_REGISTRY

MODEL = "qwen/qwen3.8-27b"

SYSTEM_PROMPT = """
You plan outdoor trips using the attraction database and weather forecast.

Use real attraction records and forecast results. Never invent data.

A suitable day must have BOTH:
- Maximum precipitation probability below 30%.
- Maximum wind speed below 35 km/h.

If no forecast day meets both rules, state that clearly.
Explain the values used. Never claim missing weather data is suitable.
The database does not provide opening hours.
Keep the final answer short.
"""

def run_agent(user_request, max_iterations=5):
    api_key = userdata.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Add GROQ_API_KEY in Colab Secrets.")

    client = Groq(api_key=api_key)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_request},
    ]
    trace = []

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.2,
            max_completion_tokens=300,
            reasoning_effort="none",
            parallel_tool_calls=False,
        )

        msg = response.choices[0].message
        calls = msg.tool_calls

        trace.append({
            "iteration": iteration + 1,
            "event": "model_response",
            "content": msg.content,
            "tool_calls": [
                {
                    "id": c.id,
                    "name": c.function.name,
                    "arguments": c.function.arguments,
                }
                for c in (calls or [])
            ],
        })

        if not calls:
            return {
                "answer": msg.content or "",
                "trace": trace,
            }

        messages.append(msg.model_dump(exclude_none=True))

        for call in calls:
            name = call.function.name

            if name not in TOOL_REGISTRY:
                result = {"error": f"Tool not approved: {name}"}
            else:
                try:
                    args = json.loads(call.function.arguments)
                    result = TOOL_REGISTRY[name].invoke(args)
                except Exception as exc:
                    result = {"error": f"{type(exc).__name__}: {exc}"}

            trace.append({
                "iteration": iteration + 1,
                "event": "tool_result",
                "tool": name,
                "result": result,
            })

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": name,
                "content": json.dumps(
                    result, ensure_ascii=False, default=str
                ),
            })

    return {
        "answer": "Tool-call limit reached. Inspect the trace and retry.",
        "trace": trace,
    }


Overwriting marathon_agent.py


In [17]:

import importlib
import marathon_agent

importlib.reload(marathon_agent)

print("Current model:", marathon_agent.MODEL)
print(
    "Current completion limit:",
    marathon_agent.run_agent.__code__.co_consts
)


Current model: qwen/qwen3.8-27b
Current completion limit: (None, 'GROQ_API_KEY', 'Add GROQ_API_KEY in Colab Secrets.', ('api_key',), 'system', ('role', 'content'), 'user', 'auto', 0.2, 300, 'none', False, ('model', 'messages', 'tools', 'tool_choice', 'temperature', 'max_completion_tokens', 'reasoning_effort', 'parallel_tool_calls'), 0, 1, 'model_response', ('id', 'name', 'arguments'), ('iteration', 'event', 'content', 'tool_calls'), '', ('answer', 'trace'), True, ('exclude_none',), 'error', 'Tool not approved: ', ': ', 'tool_result', ('iteration', 'event', 'tool', 'result'), 'tool', ('ensure_ascii', 'default'), ('role', 'tool_call_id', 'name', 'content'), 'Tool-call limit reached. Inspect the trace and retry.')


In [18]:

# Change the model in the actual file.
from pathlib import Path

path = Path("marathon_agent.py")
code = path.read_text(encoding="utf-8")

code = code.replace(
    'MODEL = "qwen/qwen3.8-27b"',
    'MODEL = "llama-3.1-8b-instant"'
)

path.write_text(code, encoding="utf-8")

# Reload the updated module.
import importlib
import marathon_agent

importlib.reload(marathon_agent)

print("Model now set to:", marathon_agent.MODEL)


Model now set to: llama-3.1-8b-instant


In [19]:

from google.colab import userdata
from groq import Groq
from pathlib import Path
import importlib
import marathon_agent

# Check models accessible to your account
client = Groq(api_key=userdata.get("GROQ_API_KEY"))
available = {model.id for model in client.models.list().data}

MODEL = "openai/gpt-oss-20b"

if MODEL not in available:
    print("That model is not available to your account.")
    print("Available models:")
    print("\n".join(sorted(available)))
else:
    path = Path("marathon_agent.py")
    code = path.read_text(encoding="utf-8")

    # Replace whichever model was previously configured.
    import re
    code = re.sub(
        r'^MODEL\s*=\s*["\'].*?["\']',
        f'MODEL = "{MODEL}"',
        code,
        count=1,
        flags=re.MULTILINE,
    )

    # Remove Qwen-specific reasoning parameter, if present.
    code = re.sub(
        r'^\s*reasoning_effort="none",\s*\n',
        '',
        code,
        flags=re.MULTILINE,
    )

    path.write_text(code, encoding="utf-8")
    importlib.reload(marathon_agent)

    print("Agent model:", marathon_agent.MODEL)


Agent model: openai/gpt-oss-20b


In [20]:
result = marathon_agent.run_agent(
    "Find one outdoor attraction in Stockholm and check the forecast. "
    "Recommend a suitable date if both weather rules are met. "
    "Keep your answer brief."
)

print("FINAL RECOMMENDATION")
print(result["answer"])

print("\nTRACE SUMMARY")
for event in result["trace"]:
    print(
        "Iteration:", event["iteration"],
        "| Event:", event["event"],
        "| Tool:", event.get("tool", "")
    )

FINAL RECOMMENDATION
**Outdoor attraction:** Djurgården (park, Stockholm)

**Suitable date:** 2026‑11‑14  
- Precipitation probability: 0 % (below 30 %)  
- Wind speed: 24.1 km/h (below 35 km/h)

(2026‑11‑15 also meets the criteria, but 14 Nov is the earliest suitable day.)

TRACE SUMMARY
Iteration: 1 | Event: model_response | Tool: 
Iteration: 1 | Event: tool_result | Tool: find_attractions_tool
Iteration: 2 | Event: model_response | Tool: 
Iteration: 2 | Event: tool_result | Tool: find_attractions_tool
Iteration: 3 | Event: model_response | Tool: 
Iteration: 3 | Event: tool_result | Tool: get_forecast_tool
Iteration: 4 | Event: model_response | Tool: 


In [21]:
print("FINAL ANSWER:")
print(result["answer"])

print("\nTRACE EVENTS:")
for event in result["trace"]:
    print(
        f"Iteration {event['iteration']} | "
        f"{event['event']} | "
        f"{event.get('tool', '')}"
    )

last_event = result["trace"][-1]
print("\nLast event:", last_event["event"])

FINAL ANSWER:
**Outdoor attraction:** Djurgården (park, Stockholm)

**Suitable date:** 2026‑11‑14  
- Precipitation probability: 0 % (below 30 %)  
- Wind speed: 24.1 km/h (below 35 km/h)

(2026‑11‑15 also meets the criteria, but 14 Nov is the earliest suitable day.)

TRACE EVENTS:
Iteration 1 | model_response | 
Iteration 1 | tool_result | find_attractions_tool
Iteration 2 | model_response | 
Iteration 2 | tool_result | find_attractions_tool
Iteration 3 | model_response | 
Iteration 3 | tool_result | get_forecast_tool
Iteration 4 | model_response | 

Last event: model_response


In [22]:
import json

called_tools = {
    event["tool"]
    for event in result["trace"]
    if event["event"] == "tool_result"
}

assert "find_attractions_tool" in called_tools
assert "get_forecast_tool" in called_tools
assert result["trace"][-1]["event"] == "model_response"

with open("stockholm_trace.json", "w", encoding="utf-8") as f:
    json.dump(
        result["trace"],
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("Step 6 checks passed.")
print("Tools used:", called_tools)
print("Trace saved to stockholm_trace.json")

Step 6 checks passed.
Tools used: {'get_forecast_tool', 'find_attractions_tool'}
Trace saved to stockholm_trace.json


##7: Investigate and evaluate the assistant

###7.1: Run test cases

In [23]:
import json
from marathon_agent import run_agent

test_requests = {
    "Stockholm": (
        "Find an outdoor attraction in Stockholm and recommend a "
        "forecast day with rain below 30% and wind below 35 km/h."
    ),
    "Berlin": (
        "Find an outdoor attraction in Berlin and recommend a "
        "forecast day with rain below 30% and wind below 35 km/h."
    ),
    "Madrid": (
        "Find an outdoor attraction in Madrid. If the database has "
        "no matching attractions, report that instead of inventing one."
    ),
    "Munich": (
        "Find an outdoor park in Munich and check the forecast. "
        "Recommend a day only if both weather thresholds are satisfied."
    ),
}

all_traces = {}
all_results = {}

for city, request in test_requests.items():
    print(f"\n{'=' * 50}")
    print(f"Testing: {city}")
    print("=" * 50)

    try:
        result = run_agent(request)
        all_results[city] = result
        all_traces[city] = result["trace"]

        print("Final answer:")
        print(result["answer"])

        tool_calls = [
            event["tool"]
            for event in result["trace"]
            if event["event"] == "tool_result"
        ]

        print("Tools called:", tool_calls)
        print("Number of tool calls:", len(tool_calls))

    except Exception as exc:
        print("Test error:", str(exc))
        all_results[city] = {"error": str(exc)}
        all_traces[city] = []

with open("marathon_traces.json", "w", encoding="utf-8") as file:
    json.dump(
        all_traces,
        file,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print("\nSaved all traces to marathon_traces.json")


Testing: Stockholm
Final answer:
**Outdoor attraction:** Djurgården (park, Stockholm) – coordinates 59.325 N, 18.103 E.  

**Suitable forecast day:** **2026‑11‑15**  
- Rain probability max: **21 %** (below 30 %)  
- Wind speed max: **23.2 km/h** (below 35 km/h)  

(2026‑11‑14 also meets the criteria, but 15 Nov is the later of the two suitable days.)
Tools called: ['find_attractions_tool', 'get_forecast_tool']
Number of tool calls: 2

Testing: Berlin
Final answer:
**Outdoor attraction:** Tempelhofer Feld (Berlin) – a large former airfield turned park.  

**Best forecast day:** 2026‑11‑12  
- Rain probability: **15 %** (below the 30 % threshold)  
- Wind speed: **12.5 km/h** (below the 35 km/h limit)  

This day meets both weather criteria, making it a suitable outing.
Tools called: ['find_attractions_tool', 'get_forecast_tool']
Number of tool calls: 2

Testing: Madrid
Final answer:
No outdoor attractions are available in the database for Madrid.
Tools called: []
Number of tool calls:

In [24]:
import pandas as pd

evaluation_table = pd.DataFrame({
    "Request": ["Stockholm", "Berlin", "Madrid", "Munich"],
    "Tools called (in order)": [
        "find_attractions_tool → find_attractions_tool → get_forecast_tool",
        "find_attractions_tool → get_forecast_tool",
        "No tools called",
        "find_attractions_tool → get_forecast_tool"
    ],
    "Calls / repeats": [
        "3 / 1",
        "2 / 0",
        "0 / 0",
        "2 / 0"
    ],
    "Correct arguments?": [
        "Yes",
        "Yes",
        "N/A",
        "Yes"
    ],
    "Final answer grounded?": [
        "Yes",
        "Yes",
        "Partially",
        "Partially"
    ],
    "Missing evidence or errors?": [
        "Repeated attraction lookup, but answer is supported by tool outputs.",
        "No significant issues detected.",
        "No database query was executed to verify the absence of attractions.",
        "Wind-speed evidence was not clearly shown in the final answer."
    ]
})

display(evaluation_table)

,Request,Tools called (in order),Calls / repeats,Correct arguments?,Final answer grounded?,Missing evidence or errors?
0,Stockholm,find_attractions_tool → find_attractions_tool ...,3 / 1,Yes,Yes,"Repeated attraction lookup, but answer is supp..."
1,Berlin,find_attractions_tool → get_forecast_tool,2 / 0,Yes,Yes,No significant issues detected.
2,Madrid,No tools called,0 / 0,N/A,Partially,No database query was executed to verify the a...
3,Munich,find_attractions_tool → get_forecast_tool,2 / 0,Yes,Partially,Wind-speed evidence was not clearly shown in t...


### 7.3 Guided Questions

### 1. Which tool was called first for each request? Why was that tool appropriate or inappropriate?

- Stockholm: `find_attractions_tool` was called first. This was appropriate because the assistant needed to find candidate attractions from the SQLite database before checking the weather.
- Berlin: `find_attractions_tool` was called first. This was appropriate because the weather API requires attraction coordinates obtained from the database.
- Madrid: No tool was called. This was not ideal because the assistant should have verified the absence of attractions through the database.
- Munich: `find_attractions_tool` was called first. This was appropriate because the request specifically asked for an outdoor park in Munich.

### 2. Did a weather-dependent recommendation use both the database and weather API? If not, which evidence is missing?

- Stockholm: Yes. The assistant used both `find_attractions_tool` and `get_forecast_tool`.
- Berlin: Yes. The assistant first retrieved attractions from the database and then checked weather conditions using the forecast tool.
- Munich: Yes. Both tools were used.
- Madrid: No weather-dependent recommendation was made. No tool evidence was provided because no tool was called.

### 3. How many times was each tool called? Did any calls repeat with identical arguments? When might repetition be justified?

- Stockholm:
  - `find_attractions_tool`: 2 times
  - `get_forecast_tool`: 1 time
  - One attraction lookup was repeated.
- Berlin:
  - `find_attractions_tool`: 1 time
  - `get_forecast_tool`: 1 time
- Madrid:
  - No tool calls.
- Munich:
  - `find_attractions_tool`: 1 time
  - `get_forecast_tool`: 1 time

A repeated call may be justified if the model needs to verify results, recover from a tool failure, or compare multiple candidate attractions before making a recommendation.

### 4. Did the final answer identify an attraction actually returned by SQLite? Did the recommended date and weather values match the Open-Meteo response?

Yes for Stockholm and Berlin.

- Stockholm recommended Djurgården, which exists in the SQLite database.
- Berlin recommended Tempelhofer Feld, which is also present in the database.
- The reported dates, precipitation probabilities, and wind speeds were obtained

# Step-8


Demonstrate one successful recommendation

In [25]:

from marathon_agent import run_agent
import json

stockholm_result = run_agent(
    "Find an outdoor attraction in Stockholm. "
    "Check the forecast and recommend a day only if rain probability "
    "is below 30% and wind speed is below 35 km/h."
)

print("FINAL ANSWER")
print(stockholm_result["answer"])

print("\nTOOL EXECUTION EVIDENCE")
for event in stockholm_result["trace"]:
    if event["event"] == "tool_result":
        print(f"\nTool: {event['tool']}")
        print(json.dumps(
            event["result"],
            indent=2,
            ensure_ascii=False,
            default=str
        ))


FINAL ANSWER
**Recommended day:** 2026‑11‑15  
**Why:**  
- Rain probability max: **21 %** (below the 30 % threshold)  
- Wind speed max: **23.2 km/h** (below the 35 km/h threshold)  

Both criteria are satisfied for this day at Djurgården, Stockholm.

TOOL EXECUTION EVIDENCE

Tool: find_attractions_tool
[]

Tool: find_attractions_tool
[
  {
    "id": 1,
    "name": "Djurgården",
    "city": "Stockholm",
    "country": "Sweden",
    "category": "park",
    "is_outdoor": 1,
    "latitude": 59.325,
    "longitude": 18.103,
    "description": "Large area for outdoor walks and sightseeing."
  },
  {
    "id": 2,
    "name": "Skansen Open-Air Museum",
    "city": "Stockholm",
    "country": "Sweden",
    "category": "museum",
    "is_outdoor": 1,
    "latitude": 59.324,
    "longitude": 18.101,
    "description": "Open-air cultural attractions and historical buildings."
  }
]

Tool: get_forecast_tool
{
  "timezone": "Europe/Stockholm",
  "retrieved_at_utc": "2026-11-12T12:43:38.890490+00:00

Demonstrate a failure or missing-data case

In [26]:

from marathon_tools import find_attractions_tool

madrid_result = find_attractions_tool.invoke({
    "city": "Madrid",
    "outdoor_only": True,
    "limit": 5
})

print("Madrid database result:")
print(madrid_result)

if not madrid_result:
    print("Expected result: no matching attractions found.")


Madrid database result:
[]
Expected result: no matching attractions found.


1. **Identify two tools that could be useful in a possible course project.**

   Two useful tools are a database search tool to find attractions and a weather forecast tool to retrieve precipitation probability and wind speed.

2. **Identify a suitable database or external API for the project.**

   SQLite is a suitable database for storing attraction names, cities, categories, coordinates, and outdoor information. The Open-Meteo Forecast API is suitable for retrieving weather forecasts.

3. **Propose one user request that requires combining at least two tools.**

   An example user request is: “Find an outdoor attraction in Stockholm and recommend a day when the rain probability is below 30% and wind speed is below 35 km/h.” This request requires the agent to use the database tool to find an attraction and the weather tool to check the forecast.

4. **Describe one potential tool-use failure and an evaluation method for detecting it.**

   One potential failure is that the database may not contain attractions for the requested city, or the weather API may fail to return forecast data. To detect these failures, inspect the tool execution trace, check whether the tools returned valid results, and verify that the final recommendation is supported by the database and weather data. Both weather thresholds must be satisfied on the same date.

#Code Marathon 2: Weather and Trip Planner with MCP

In [27]:

%pip install -q "mcp>=1.28,<2" langchain langchain-mcp-adapters langchain-groq langgraph requests


In [28]:
%%writefile marathon_mcp_server.py

from mcp.server.fastmcp import FastMCP

from marathon_db import (
    initialize_db,
    find_attractions
)

from marathon_api import (
    get_forecast
)

initialize_db()

mcp = FastMCP("WeatherTripPlanner")


@mcp.tool()
def find_attractions_tool(
    city: str,
    category: str = None,
    outdoor_only: bool = True,
    limit: int = 5
):
    """
    Search attractions in the local SQLite database.
    """
    return find_attractions(
        city=city,
        category=category,
        outdoor_only=outdoor_only,
        limit=limit
    )


@mcp.tool()
def get_forecast_tool(
    latitude: float,
    longitude: float,
    days: int = 7
):
    """
    Get weather forecast from Open-Meteo.
    """
    return get_forecast(
        latitude,
        longitude,
        days
    )


if __name__ == "__main__":
    mcp.run()

Overwriting marathon_mcp_server.py


In [29]:
!ls

attractions_sweden_germany.sqlite  marathon_mcp_server.py  sample_data
marathon_agent.py		   marathon_tools.py	   stockholm_trace.json
marathon_api.py			   marathon_traces.json    test_mcp_client.py
marathon_db.py			   __pycache__


In [30]:
!python -m py_compile marathon_mcp_server.py

In [ ]:
!python marathon_mcp_server.py

In [ ]:
from mcp.server.fastmcp import FastMCP

print("FastMCP OK")

In [ ]:
%%writefile test_mcp_client.py

import asyncio

from mcp import ClientSession
from mcp.client.stdio import stdio_client
from mcp.client.stdio import StdioServerParameters


async def main():

    print("Creating server params...")

    server_params = StdioServerParameters(
        command="python3",
        args=["marathon_mcp_server.py"],
    )

    print("Connecting...")

    async with stdio_client(server_params) as (
        read_stream,
        write_stream,
    ):

        print("Connected")

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            print("Initializing")

            await session.initialize()

            print("Initialized\n")

            tools = await session.list_tools()

            print("TOOLS FOUND:")
            print(tools)


asyncio.run(main())


In [ ]:
!python test_mcp_client.py

In [ ]:
from marathon_mcp_server import mcp

print(type(mcp))
print(dir(mcp))

### Stage 4: Test LangChain Integration with MCPAdapter (No LLM)
We will use `langchain_mcp_adapters` to connect to our local MCP server over stdio, retrieve the tools, and invoke them directly using LangChain's interface to ensure the adapter layer works correctly.

In [ ]:
import asyncio
import sys
from langchain_mcp_adapters.tools import load_mcp_tools
from mcp.client.stdio import stdio_client
from mcp import StdioServerParameters, ClientSession

async def test_adapted_tools():
    # Use sys.executable to ensure we utilize the exact same python path
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["marathon_mcp_server.py"],
    )

    print("Connecting to MCP Server and loading tools into LangChain adapter...")

    try:
        # Specify errlog=sys.__stderr__ to bypass Colab stream issues
        async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
            async with ClientSession(read, write) as session:
                print("Initializing session...")
                await session.initialize()

                print("Loading adapted tools...")
                lc_tools = await load_mcp_tools(session)

                print(f"\nSuccessfully loaded {len(lc_tools)} tools:")
                for tool in lc_tools:
                    print(f" - Name: {tool.name}")
                    print(f"   Description: {tool.description.strip()}")

                # Test find_attractions_tool
                find_tool = next(t for t in lc_tools if t.name == "find_attractions_tool")
                print("\nTesting find_attractions_tool directly...")
                attraction_result = await find_tool.ainvoke({"city": "Stockholm", "outdoor_only": True, "limit": 2})
                print("Result:", attraction_result)

                # Test get_forecast_tool
                forecast_tool = next(t for t in lc_tools if t.name == "get_forecast_tool")
                print("\nTesting get_forecast_tool directly...")
                forecast_result = await forecast_tool.ainvoke({"latitude": 59.325, "longitude": 18.103, "days": 3})
                print("Result:", str(forecast_result)[:300] + "...")
    except Exception as e:
        print(f"Error during MCP Adapter execution: {e}")
        import traceback
        traceback.print_exc(file=sys.stdout)

await test_adapted_tools()

In [ ]:
import subprocess
try:
    result = subprocess.run(
        ["python3", "marathon_mcp_server.py"],
        capture_output=True,
        text=True,
        timeout=5
    )
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
except subprocess.TimeoutExpired as e:
    print("Server started successfully (it didn't crash within 5 seconds)")
    print("Captured STDOUT:", e.stdout.decode() if e.stdout else "")
    print("Captured STDERR:", e.stderr.decode() if e.stderr else "")
except Exception as e:
    print("Error running subprocess:", e)

In [ ]:
import subprocess
# Test running the MCP server directly using python to capture its startup errors (stderr)
try:
    result = subprocess.run(
        ["python3", "marathon_mcp_server.py"],
        capture_output=True,
        text=True,
        timeout=5
    )
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
except subprocess.TimeoutExpired as e:
    print("Server started successfully (it didn't crash within 5 seconds)")
    print("Captured STDOUT:", e.stdout.decode() if e.stdout else "")
    print("Captured STDERR:", e.stderr.decode() if e.stderr else "")
except Exception as e:
    print("Error running subprocess:", e)

### Stage 5: Build LangChain Agent with MCP Tools & Groq LLM
We will now write an agent script/function that utilizes the MCP tools and a Groq-hosted model to recommend suitable days for outdoor visits based on the 30% precipitation and 35 km/h wind rules.

In [ ]:
%%writefile marathon2_agent.py
import os
import sys
import json
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_mcp_adapters.tools import load_mcp_tools
from mcp.client.stdio import stdio_client
from mcp import StdioServerParameters, ClientSession
from langgraph.prebuilt import create_react_agent

# Set Groq API Key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

SYSTEM_PROMPT = """
You plan outdoor trips using the attraction database and weather forecast provided via your MCP tools.

Use real attraction records and forecast results. Never invent data.

To recommend a suitable day, it must satisfy BOTH:
- Maximum precipitation probability below 30%.
- Maximum wind speed below 35 km/h.

If no forecast day meets both rules, state that clearly.
Explain the values used. Never claim missing weather data is suitable.
The database does not provide opening hours.
Keep the final answer short and grounded in tool evidence.
"""

async def run_mcp_agent(user_request, model_name="openai/gpt-oss-20b"):
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["marathon_mcp_server.py"],
    )

    async with stdio_client(server_params, errlog=sys.__stderr__) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)

            llm = ChatGroq(
                model=model_name,
                temperature=0.2
            )

            agent_executor = create_react_agent(
                model=llm,
                tools=tools,
                state_modifier=SYSTEM_PROMPT
            )

            inputs = {"messages": [("user", user_request)]}
            trace = []
            final_answer = ""

            async for event in agent_executor.astream(inputs, stream_mode="values"):
                message_list = event.get("messages", [])
                if message_list:
                    last_msg = message_list[-1]
                    if last_msg.type == "ai":
                        trace.append({
                            "role": "assistant",
                            "content": last_msg.content,
                            "tool_calls": last_msg.tool_calls
                        })
                        final_answer = last_msg.content
                    elif last_msg.type == "tool":
                        trace.append({
                            "role": "tool",
                            "name": last_msg.name,
                            "content": last_msg.content
                        })

            return {
                "answer": final_answer,
                "trace": trace
            }

### Stage 6: Evaluate & Run 4-City Test Suite
We will execute the agent for Stockholm, Berlin, Madrid, and Munich, outputting the structured execution traces into `marathon2_traces.json` to prove groundedness.

In [ ]:
import json
import pandas as pd
from marathon2_agent import run_mcp_agent

test_requests = {
    "Stockholm": (
        "Find an outdoor attraction in Stockholm and recommend a "
        "forecast day with rain below 30% and wind below 35 km/h."
    ),
    "Berlin": (
        "Find an outdoor attraction in Berlin and recommend a "
        "forecast day with rain below 30% and wind below 35 km/h."
    ),
    "Madrid": (
        "Find an outdoor attraction in Madrid. If the database has "
        "no matching attractions, report that instead of inventing one."
    ),
    "Munich": (
        "Find an outdoor park in Munich and check the forecast. "
        "Recommend a day only if both weather thresholds are satisfied."
    ),
}

mcp_traces = {}
mcp_results = {}

for city, request in test_requests.items():
    print(f"\n{'=' * 50}")
    print(f"Testing MCP Agent for: {city}")
    print("=" * 50)
    try:
        # Run the async agent function in the current notebook event loop
        result = await run_mcp_agent(request, model_name="openai/gpt-oss-20b")
        mcp_results[city] = result["answer"]
        mcp_traces[city] = result["trace"]

        print("\nFinal Answer:")
        print(result["answer"])
    except Exception as e:
        print(f"Error processing {city}: {e}")
        mcp_results[city] = f"Error: {e}"
        mcp_traces[city] = []

# Save traces
with open("marathon2_traces.json", "w", encoding="utf-8") as f:
    json.dump(mcp_traces, f, indent=2, ensure_ascii=False)

print("\nAll traces successfully saved to marathon2_traces.json")

### Stage 7: External MCP Challenge (Prerequisite Week3-Activity-1)

#### Part 1: Connect and Test the Remote HTTP Server (Without an LLM)

In [ ]:
import json
from mcp import Client

# Using the endpoint provided in Week 3 Activity 1
PARALLEL_HTTP = "https://search.parallel.ai/mcp"
DOC_URL = "https://modelcontextprotocol.io/introduction"

async def complete_stage_7():
    try:
        # Connect using the HTTP endpoint
        async with Client(PARALLEL_HTTP) as client:

            # 1. Discover the schema
            page = await client.list_tools()
            remote_registry = {tool.name: tool for tool in page.tools}

            print("=== DISCOVERED SCHEMA ===")
            if "web_fetch" in remote_registry:
                print(json.dumps(remote_registry["web_fetch"].input_schema, indent=2))

                # 2. Test the read-only capability
                print("\n=== ATTEMPTING TOOL CALL ===")
                result = await client.call_tool(
                    "web_fetch",
                    {"urls": [DOC_URL], "objective": "Explain the purpose of MCP"}
                )

                print("Error Status:", result.is_error)
                if not result.is_error:
                     print("Result Snippet:", result.content[0].text[:500])
                else:
                     print("Failure:", result.content)
            else:
                print("web_fetch tool was not advertised by the server.")

    except Exception as exc:
        print("\n=== CONNECTION EXCEPTION ===")
        print(f"{type(exc).__name__}: {str(exc)[:300]}")

await complete_stage_7()

#### Part 3: Recording Findings (The Deliverable Checklist)




1. Endpoint(https://search.parallel.ai/mcp)

2. Discovered Schema:
> A read-only schema named web_fetch. Its JSON input schema requires a urls array and an objective string.

3. Results or Failures:(work in progress)


4. Transport Differences:
> **Streamable HTTP (Remote)**: The client connects over a network to an independent, already-runnning service via a URL, exchanging JSON-RPC protocol messages over HTTP requests and responses.  
**stdio(local):** The clinet acts as a parent process that actively launches the server as a local child subprocess. The MCP messages travel locally through stdin and stdout pipes, requiring no network ports for the MCP link.